# Optativa III: Ciencia de Datos
## Semana 4 — Procesamiento y limpieza de datos (Data Wrangling con pandas)

**Universidad de Especialidades UNE · Plantel Centro**
**Ingeniería en Computación · 9.º semestre · Semana 4 · Sesión 1**

---

### Propósito

En ciencia de datos, entre el **70% y el 80% del tiempo** se dedica a limpiar y preparar datos. Un modelo brillante sobre datos sucios produce basura. En esta actividad aprenderás **data wrangling**: cargar datos desde **múltiples fuentes** (CSV, JSON, SQL) y resolver los problemas más comunes: **valores nulos, duplicados, tipos de datos incorrectos, formato inconsistente y valores atípicos**.

La actividad tiene dos partes:

- **PARTE A — Ejemplo guiado por el profesor.** Limpiaremos juntos, paso a paso, un dataset "sucio" de ventas. Observa cada técnica y responde las preguntas.
- **PARTE B — Tu turno.** Aplicarás lo aprendido de forma autónoma sobre datos de clientes (JSON) y productos (SQL), completando el código.

Responderás **50 preguntas ✍️** en total, modificando el código, analizando resultados e interpretando.

### Los datasets (súbelos a Colab antes de empezar)

- `ventas_sucias.csv` — ventas de una tienda, con muchos problemas de calidad.
- `clientes.json` — datos de clientes en formato JSON.
- `tienda.db` — base de datos SQLite con una tabla de productos.

Sube los tres archivos con el panel de archivos de Colab (icono de carpeta → subir).

### Cómo trabajar

Ejecuta cada celda con **Shift + Enter** en orden, responde las preguntas ✍️ (doble clic), completa los retos y, al final, guarda una copia en Drive y entrégala en Classroom.


---
# PARTE A — Ejemplo guiado por el profesor

## Paso 0 — Cargar librerías


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import sqlite3

pd.set_option("display.max_columns", None)
print("Librerias cargadas.")

> ✍️ **1.** ¿Por qué se dice que la limpieza de datos consume la mayor parte del tiempo de un proyecto de ciencia de datos? ¿Qué pasaría si entrenáramos un modelo sin limpiar los datos?


✍️ **1. Respuesta:**

La limpieza de datos consume la mayor parte del tiempo en un proyecto de ciencia de datos (70-80%) porque los datos del mundo real rara vez son perfectos. Vienen de diversas fuentes, con errores humanos, inconsistencias, valores faltantes, formatos incorrectos y ruidos. Transformar estos datos "sucios" en un formato utilizable y confiable requiere un esfuerzo considerable.

Si entrenáramos un modelo sin limpiar los datos, el resultado sería un "basura adentro, basura afuera" (garbage in, garbage out). Un modelo entrenado con datos sucios:

*   **Rendirá mal:** No generalizará bien a datos nuevos y producirá predicciones inexactas o sin sentido.
*   **Será inestable:** Pequeñas variaciones en los datos de entrada podrían llevar a grandes cambios en el resultado.
*   **Sacará conclusiones erróneas:** Las estadísticas, relaciones y patrones identificados serían engañosos, llevando a decisiones empresariales incorrectas.
*   **Será difícil de interpretar:** La presencia de errores oscurecería los verdaderos patrones, haciendo que el modelo sea difícil de entender y depurar.

## Paso 1 — Carga desde CSV y primer diagnóstico

Cargamos el dataset de ventas y hacemos una **auditoría inicial** de su calidad.


In [ ]:
df = pd.read_csv("ventas_sucias - ventas_sucias.csv")
print("Dimensiones:", df.shape)
df.head(10)

In [ ]:
# Diagnostico de calidad: tipos, nulos y duplicados
print("=== TIPOS DE DATOS ===")
print(df.dtypes)
print("\n=== VALORES NULOS POR COLUMNA ===")
print(df.isnull().sum())
print("\n=== FILAS DUPLICADAS ===")
print("Duplicados exactos:", df.duplicated().sum())

> ✍️ **2.** ¿Cuántas filas y columnas tiene el dataset original? Ejecuta `df.shape`.

> ✍️ **3.** ¿Qué columnas tienen valores nulos y cuál es la que más tiene? Reporta las cifras.

> ✍️ **4.** ¿Cuántas filas duplicadas exactas hay? ¿Por qué los duplicados son un problema al calcular estadísticas como promedios o totales?

> ✍️ **5.** Observa `df.dtypes`. La columna `precio_total` NO es numérica (aparece como `object`). ¿Por qué crees que pandas la cargó como texto? (Pista: mira sus valores en `df.head()`.)


## Paso 2 — Eliminar duplicados


In [ ]:
filas_antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
filas_despues = len(df)
print(f"Filas antes:   {filas_antes}")
print(f"Filas despues: {filas_despues}")
print(f"Duplicados eliminados: {filas_antes - filas_despues}")

> ✍️ **6.** ¿Cuántas filas se eliminaron al quitar duplicados? ¿Por qué usamos `reset_index(drop=True)` después?

> ✍️ **7.** El método `drop_duplicates()` tiene un parámetro `subset`. Investiga: ¿cómo eliminarías duplicados basándote SOLO en la columna `id_venta` en lugar de en filas completas? Escríbelo en una celda (no lo ejecutes sobre df todavía).


In [ ]:
# Celda para la P7: ejemplo de drop_duplicates con subset (solo demuestra, usa una copia)
# ejemplo = df.drop_duplicates(subset=["id_venta"])
# print(ejemplo.shape)


## Paso 3 — Corregir tipos de datos (texto → número)

La columna `precio_total` tiene valores como `"$450.5"`. Debemos quitar el símbolo y convertirla a número.


In [ ]:
# Antes: ver algunos valores problematicos
print("Valores de ejemplo (antes):", df["precio_total"].head(8).tolist())

# Limpieza: convertir a texto, quitar '$' y espacios, convertir a float
df["precio_total"] = (df["precio_total"]
                      .astype(str)
                      .str.replace("$", "", regex=False)
                      .str.strip())
df["precio_total"] = pd.to_numeric(df["precio_total"], errors="coerce")

print("Tipo despues:", df["precio_total"].dtype)
print("Media de precio_total:", round(df["precio_total"].mean(), 2))

> ✍️ **8.** ¿Qué hace `pd.to_numeric(..., errors="coerce")`? ¿Qué pasa con los valores que no se pueden convertir?

> ✍️ **9.** Explica cada paso de la limpieza: ¿por qué primero `.astype(str)`, luego `.str.replace("$","")` y luego `.str.strip()`?

> ✍️ **10.** Ahora que `precio_total` es numérica, ¿pudo aparecer algún nuevo valor nulo? Ejecuta `df["precio_total"].isnull().sum()` en una celda y explica por qué.


In [ ]:
# Celda para la P10
# print(df["precio_total"].isnull().sum())


## Paso 4 — Normalizar texto inconsistente

Las columnas `pais` y `categoria` tienen el mismo valor escrito de muchas formas: "México", "MEXICO", "  México ", "mexico". Hay que **estandarizarlas**.


In [ ]:
print("Paises unicos ANTES de limpiar:")
print(df["pais"].dropna().unique())

# Normalizamos: quitar espacios, poner en minusculas, capitalizar
df["pais"] = df["pais"].str.strip().str.lower()

# Unificar equivalencias con un diccionario de reemplazo
mapa_pais = {
    "mexico": "México", "méxico": "México", "mx": "México",
    "colombia": "Colombia",
    "españa": "España",
    "peru": "Perú", "perú": "Perú",
}
df["pais"] = df["pais"].replace(mapa_pais)

print("\nPaises unicos DESPUES de limpiar:")
print(df["pais"].dropna().unique())

> ✍️ **11.** ¿Cuántos valores distintos de `pais` había antes y cuántos quedaron después? ¿Por qué es importante unificarlos?

> ✍️ **12.** Explica qué hace la cadena `.str.strip().str.lower()`. ¿Por qué el orden importa?

> ✍️ **13.** El diccionario `mapa_pais` unifica variantes. ¿Qué pasaría en un análisis de "ventas por país" si NO hubiéramos unificado "mexico", "México" y "MX"?


### 🔧 Reto 1 — Limpia la columna `categoria`

Aplica la misma técnica (strip + lower + diccionario de reemplazo) para unificar la columna `categoria`, que tiene "Electrónica", "electronica", "Electronica ", etc.


In [ ]:
# RETO: normaliza df["categoria"]
# print("Antes:", df["categoria"].dropna().unique())
# df["categoria"] = df["categoria"].str.strip().str.lower()
# mapa_cat = {"electronica": "Electrónica", "electrónica": "Electrónica", ...}
# df["categoria"] = df["categoria"].replace(mapa_cat)
# print("Despues:", df["categoria"].dropna().unique())


> ✍️ **14.** ¿Cuántas categorías únicas quedaron tras tu limpieza? Lístalas.

> ✍️ **15.** ¿Qué riesgo tiene un diccionario de reemplazo escrito a mano si el dataset tuviera miles de variantes distintas? Propón una idea de cómo automatizarlo.


## Paso 5 — Manejo de valores nulos

Tenemos nulos en varias columnas. Hay distintas estrategias: **eliminar** filas, **rellenar** (imputar) con la media/mediana/moda, o dejar según el caso.


In [ ]:
print("Nulos actuales por columna:")
print(df.isnull().sum())
print()

# Convertir precio_unitario a numero antes de imputar valores faltantes o invalidos
df["precio_unitario"] = pd.to_numeric(df["precio_unitario"], errors="coerce")

# Estrategia 1: imputar precio_unitario con la MEDIANA (robusta ante outliers)
mediana_precio = df["precio_unitario"].median()
df["precio_unitario"] = df["precio_unitario"].fillna(mediana_precio)
print(f"precio_unitario: nulos rellenados con la mediana ({mediana_precio:.2f})")

# Estrategia 2: imputar categoria y pais con la MODA (valor mas frecuente)
for col in ["pais", "categoria", "metodo_pago"]:
    if df[col].isnull().sum() > 0:
        moda = df[col].mode()[0]
        df[col] = df[col].fillna(moda)
        print(f"{col}: nulos rellenados con la moda ('{moda}')")

print("\nNulos despues de imputar:")
print(df.isnull().sum())

> ✍️ **16.** ¿Por qué imputamos `precio_unitario` con la **mediana** y no con la media? (Recuerda la lección de robustez de la Semana 2.)

> ✍️ **17.** ¿Por qué para `pais` y `categoria` (variables categóricas) usamos la **moda** en lugar de la media? ¿Tendría sentido calcular la "media" de un país?

> ✍️ **18.** ¿Cuándo sería mejor **eliminar** las filas con nulos en vez de imputarlas? Piensa en el porcentaje de datos faltantes.

> ✍️ **19.** La imputación introduce un sesgo: rellenamos con valores "inventados". ¿Qué desventaja tiene imputar muchos nulos con la misma mediana o moda?


### 🔧 Reto 2 — Decide qué hacer con `fecha`

La columna `fecha` tiene nulos y formatos mezclados. Cuenta cuántos nulos tiene y decide una estrategia (eliminar esas filas o marcarlas). Justifica tu decisión en la pregunta.


In [ ]:
# RETO: analiza los nulos de fecha
# print("Nulos en fecha:", df["fecha"].isnull().sum())
# print("Formatos de ejemplo:", df["fecha"].dropna().unique()[:6])


> ✍️ **20.** ¿Cuántos nulos tiene `fecha`? ¿Qué estrategia elegiste y por qué? ¿Es fácil imputar una fecha faltante?


## Paso 6 — Corregir valores imposibles

La columna `edad_cliente` tiene valores imposibles (negativos como −5, o 200 años) y `calificacion` tiene un valor inválido (99, cuando debería ser 1-5).


In [ ]:
print("Edades unicas antes:", sorted(df["edad_cliente"].dropna().unique()))
print("Calificaciones unicas antes:", sorted(df["calificacion"].dropna().unique()))

# Reemplazamos valores imposibles por NaN (fuera de rango valido)
df.loc[(df["edad_cliente"] < 0) | (df["edad_cliente"] > 120), "edad_cliente"] = np.nan
df.loc[(df["calificacion"] < 1) | (df["calificacion"] > 5), "calificacion"] = np.nan

# Luego imputamos con la mediana
df["edad_cliente"] = df["edad_cliente"].fillna(df["edad_cliente"].median())
df["calificacion"] = df["calificacion"].fillna(df["calificacion"].median())

print("\nEdades unicas despues:", sorted(df["edad_cliente"].dropna().unique()))
print("Calificaciones unicas despues:", sorted(df["calificacion"].dropna().unique()))

> ✍️ **21.** ¿Qué valores imposibles había en `edad_cliente` y `calificacion`? ¿Por qué primero los convertimos a NaN antes de imputar?

> ✍️ **22.** ¿Por qué un valor de 99 en una calificación de 1 a 5 es peligroso si NO lo corregimos? ¿Cómo afectaría al promedio de calificaciones?

> ✍️ **23.** Define "regla de negocio" para la validación de datos. ¿Qué regla aplicaste para la edad y cuál para la calificación?


## Paso 7 — Detectar y tratar valores atípicos (outliers)

La columna `precio_unitario` tiene valores extremos (inyectamos precios 50 veces mayores). Los detectamos con el método del **rango intercuartílico (IQR)**.


In [ ]:
Q1 = df["precio_unitario"].quantile(0.25)
Q3 = df["precio_unitario"].quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df[(df["precio_unitario"] < limite_inferior) | (df["precio_unitario"] > limite_superior)]
print(f"Q1={Q1:.2f}  Q3={Q3:.2f}  IQR={IQR:.2f}")
print(f"Limites normales: [{limite_inferior:.2f}, {limite_superior:.2f}]")
print(f"Numero de outliers detectados: {len(outliers)}")

# Visualizamos con un boxplot antes de tratar
fig, axes = plt.subplots(1, 2, figsize=(13,5))
axes[0].boxplot(df["precio_unitario"])
axes[0].set_title("Precio unitario CON outliers"); axes[0].set_ylabel("Precio")

# Tratamiento: recortar (capping) al limite superior
df["precio_unitario"] = df["precio_unitario"].clip(upper=limite_superior)

axes[1].boxplot(df["precio_unitario"])
axes[1].set_title("Precio unitario DESPUES (capping)")
plt.tight_layout(); plt.show()

> ✍️ **24.** ¿Cuántos outliers detectó el método IQR en `precio_unitario`? ¿Cuál es el límite superior calculado?

> ✍️ **25.** Explica el método IQR con tus palabras: ¿por qué se usa 1.5 × IQR como umbral?

> ✍️ **26.** Compara los dos boxplots. ¿Cómo cambió la caja tras el "capping"? ¿Qué hace el método `.clip(upper=...)`?

> ✍️ **27.** El "capping" recorta los valores extremos al límite. Menciona otra estrategia posible para tratar outliers (por ejemplo, eliminarlos) y di cuándo sería preferible cada una.

> ✍️ **28.** ¿Todos los outliers son errores? Da un ejemplo de un outlier que sea un dato REAL y valioso (no un error) que NO deberíamos eliminar.


## Paso 8 — Transformar y crear variables (feature engineering)

Con los datos limpios, creamos variables nuevas útiles para el análisis.


In [ ]:
# Verificar la coherencia: precio_total deberia ser precio_unitario * cantidad
df["precio_total_calculado"] = (df["precio_unitario"] * df["cantidad"]).round(2)

# Crear una variable categorica: rango de precio
df["rango_precio"] = pd.cut(df["precio_unitario"],
                            bins=[0, 200, 500, 1000, np.inf],
                            labels=["Bajo", "Medio", "Alto", "Premium"])

print("Distribucion por rango de precio:")
print(df["rango_precio"].value_counts())

df[["precio_unitario", "cantidad", "precio_total_calculado", "rango_precio"]].head()

> ✍️ **29.** ¿Qué hace `pd.cut()`? ¿En qué se diferencia una variable continua (`precio_unitario`) de la variable categórica que creamos (`rango_precio`)?

> ✍️ **30.** ¿Por qué crear la columna `precio_total_calculado` es útil para verificar la calidad de los datos? (Pista: compárala con `precio_total`.)


## Paso 9 — Dataset limpio: verificación final


In [ ]:
print("=== RESUMEN DE LIMPIEZA ===")
print(f"Filas finales: {len(df)}")
print(f"Nulos totales restantes: {df.isnull().sum().sum()}")
print(f"Duplicados: {df.duplicated().sum()}")
print(f"Paises unicos: {df['pais'].nunique()}")
print(f"Categorias unicas: {df['categoria'].nunique()}")
print()
print("Estadisticas del dataset limpio:")
df[["precio_unitario", "cantidad", "precio_total", "calificacion", "edad_cliente"]].describe()

> ✍️ **31.** Compara el dataset final con el inicial: ¿cuántos nulos quedan? ¿Cuántos países y categorías únicas? Resume en 3 líneas qué logró la limpieza.

> ✍️ **32.** Grafica la distribución de ventas por categoría con `df["categoria"].value_counts().plot(kind="bar")` en una celda. ¿Qué categoría vende más?


In [ ]:
# Celda para la P32: grafico de barras de categorias
# df["categoria"].value_counts().plot(kind="bar", color="#1e2a6b")
# plt.title("Ventas por categoria"); plt.ylabel("Numero de ventas"); plt.show()


---
# PARTE B — Tu turno (trabajo autónomo)

Ahora aplicarás lo aprendido a **otras fuentes de datos**. El profesor ya no te da el código completo: debes escribirlo tú.

## Paso 10 — Cargar datos desde JSON

Los datos de clientes están en formato **JSON**. Cárgalos y diagnostica su calidad.


In [ ]:
# Cargar el JSON de clientes
with open("clientes.json", "r", encoding="utf-8") as f:
    datos_json = json.load(f)

clientes = pd.DataFrame(datos_json)
print("Clientes cargados:", clientes.shape)
clientes.head()

> ✍️ **33.** ¿Cuántos clientes y cuántas columnas hay? ¿Qué diferencia hay entre cargar un CSV y un JSON en cuanto a estructura?

> ✍️ **34.** Ejecuta `clientes.isnull().sum()`. ¿Qué columnas tienen nulos?

> ✍️ **35.** Observa la columna `nombre`: tiene valores como `" ana "`, `"LUIS"`, `""` (vacío). ¿Qué problemas de calidad identificas?


In [ ]:
# Celda para explorar nulos y valores de clientes
# print(clientes.isnull().sum())
# print(clientes["nombre"].unique())
# print(clientes["email"].unique())


### 🔧 Reto 3 — Limpia la columna `nombre`

Normaliza `nombre`: quita espacios, capitaliza (primera letra mayúscula) y reemplaza los vacíos `""` por NaN.


In [ ]:
# RETO: limpia clientes["nombre"]
# clientes["nombre"] = clientes["nombre"].str.strip().str.capitalize()
# clientes["nombre"] = clientes["nombre"].replace("", np.nan)
# print(clientes["nombre"].unique())


> ✍️ **36.** ¿Qué hace `.str.capitalize()`? Muestra los nombres únicos después de limpiar.

> ✍️ **37.** La columna `email` tiene valores como `"sin_correo"` y `None`. Escribe código para contar cuántos emails son inválidos (que no contengan "@"). ¿Cuántos hay?


In [ ]:
# Celda para la P37: contar emails invalidos
# invalidos = clientes["email"].fillna("").str.contains("@") == False
# print("Emails invalidos:", invalidos.sum())


### 🔧 Reto 4 — Normaliza la columna `ciudad`

La columna `ciudad` tiene "CDMX", "cdmx", "Guadalajara", "GDL", etc. Unifícala con un diccionario de reemplazo, como hicimos con `pais`.


In [ ]:
# RETO: normaliza clientes["ciudad"]
# clientes["ciudad"] = clientes["ciudad"].str.strip().str.upper()
# mapa_ciudad = {"CDMX": "Ciudad de México", "GDL": "Guadalajara", ...}
# clientes["ciudad"] = clientes["ciudad"].replace(mapa_ciudad)
# print(clientes["ciudad"].value_counts())


> ✍️ **38.** ¿Cuántas ciudades únicas quedaron tras normalizar? ¿Qué variantes unificaste?

> ✍️ **39.** Calcula el `gasto_total` promedio por ciudad con `clientes.groupby("ciudad")["gasto_total"].mean()`. ¿Qué ciudad gasta más en promedio?


In [ ]:
# Celda para la P39: gasto promedio por ciudad
# print(clientes.groupby("ciudad")["gasto_total"].mean().round(2))


## Paso 11 — Cargar datos desde SQL (SQLite)

Los productos están en una base de datos SQL. Se consultan con lenguaje SQL a través de pandas.


In [ ]:
# Conectar a la base de datos y consultar
conn = sqlite3.connect("tienda.db")

# Consulta SQL: traer todos los productos
productos = pd.read_sql("SELECT * FROM productos", conn)
print("Productos cargados:", productos.shape)
print(productos.head())

conn.close()

> ✍️ **40.** ¿Cuántos productos hay en la base de datos? ¿Qué columnas tiene la tabla `productos`?

> ✍️ **41.** ¿Qué ventaja tiene almacenar datos en una base SQL frente a un archivo CSV, para un negocio con millones de registros?


### 🔧 Reto 5 — Consulta SQL con filtro

Modifica la consulta para traer **solo los productos con stock menor a 50** (productos por reabastecer). Usa `WHERE stock < 50`.


In [ ]:
# RETO: consulta SQL con filtro
# conn = sqlite3.connect("tienda.db")
# bajo_stock = pd.read_sql("SELECT * FROM productos WHERE stock < 50", conn)
# conn.close()
# print("Productos con bajo stock:", len(bajo_stock))


> ✍️ **42.** ¿Cuántos productos tienen stock menor a 50? ¿Por qué filtrar en la consulta SQL es más eficiente que cargar todo y filtrar en pandas, cuando la base es enorme?

> ✍️ **43.** Escribe una consulta SQL que traiga el **precio promedio por categoría** usando `SELECT categoria, AVG(precio) FROM productos GROUP BY categoria`. ¿Qué categoría es la más cara?


In [ ]:
# RETO/Celda para la P43: precio promedio por categoria con SQL
# conn = sqlite3.connect("tienda.db")
# resultado = pd.read_sql("SELECT categoria, AVG(precio) as precio_prom FROM productos GROUP BY categoria", conn)
# conn.close()
# print(resultado)


## Paso 12 — Combinar fuentes (merge)

El verdadero poder aparece al **unir** datos de distintas fuentes. Unamos las ventas limpias con información adicional.


In [ ]:
# Ejemplo: cuantas ventas hay por categoria (del CSV limpio)
ventas_por_cat = df.groupby("categoria").agg(
    num_ventas=("id_venta", "count"),
    precio_promedio=("precio_unitario", "mean")
).round(2)
print("Resumen de ventas por categoria (dataset limpio):")
print(ventas_por_cat)

> ✍️ **44.** ¿Qué hace el método `groupby().agg()`? Explica qué calcula cada línea del `agg`.

> ✍️ **45.** Investiga brevemente: ¿qué es un `merge` (o `join`) en pandas y para qué serviría unir la tabla de ventas con la de clientes? Da un ejemplo de qué pregunta podrías responder al unirlas.


## Paso 13 — Análisis final e interpretación


In [ ]:
# Un analisis integral sobre los datos limpios
print("=== ANALISIS DEL DATASET LIMPIO ===")
print(f"\nVenta promedio: ${df['precio_total'].mean():.2f}")
print(f"Calificacion promedio: {df['calificacion'].mean():.2f}")
print(f"Edad promedio del cliente: {df['edad_cliente'].mean():.1f} años")
print(f"\nMetodo de pago mas usado: {df['metodo_pago'].mode()[0]}")
print(f"Pais con mas ventas: {df['pais'].mode()[0]}")

# Grafico resumen
fig, axes = plt.subplots(1, 2, figsize=(13,5))
df["categoria"].value_counts().plot(kind="bar", ax=axes[0], color="#1e2a6b")
axes[0].set_title("Ventas por categoria"); axes[0].set_ylabel("Numero de ventas")
df["metodo_pago"].value_counts().plot(kind="pie", ax=axes[1], autopct="%1.0f%%")
axes[1].set_title("Metodos de pago"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()

> ✍️ **46.** Según el análisis, ¿cuál es la venta promedio, la calificación promedio y el método de pago más usado?

> ✍️ **47.** ¿Habría sido posible hacer este análisis correctamente SIN limpiar los datos primero? Explica qué habría salido mal en cada estadística (por ejemplo, el promedio de precio con los outliers).

> ✍️ **48.** De todas las técnicas de limpieza que aplicaste (duplicados, tipos, nulos, texto, outliers, valores imposibles), ¿cuál te pareció más importante para este dataset y por qué?

> ✍️ **49.** Un compañero dice: "Limpiar datos es aburrido y no es parte 'real' de la ciencia de datos". Argumenta por qué está equivocado, con ejemplos de esta actividad.

> ✍️ **50 (reflexión).** En tu carrera de Ingeniería en Computación, ¿en qué situación real tendrías que limpiar datos sucios? (Por ejemplo: logs de un servidor, datos de sensores IoT, registros de usuarios). Describe qué problemas de calidad esperarías encontrar.


---
## Cierre y entrega

Aprendiste a cargar datos desde **tres fuentes** (CSV, JSON, SQL) y aplicaste el flujo completo de **data wrangling**: diagnóstico, duplicados, tipos, texto inconsistente, nulos, valores imposibles, outliers y creación de variables. Sobre todo, comprobaste que **sin datos limpios no hay análisis confiable**.

### Entrega en Google Classroom

1. Verifica que **todas las celdas corran sin error** (Entorno de ejecución → Ejecutar todo).
2. Confirma que respondiste las **50 preguntas ✍️** y los **5 retos de código**.
3. **Archivo → Guardar una copia en Drive** y comparte el enlace en Classroom.

### Rúbrica

| Criterio | Puntos |
|---|---|
| Todas las celdas se ejecutan correctamente | 10 |
| Retos de código resueltos (5) | 25 |
| Respuestas a las 50 preguntas ✍️ | 45 |
| Calidad del análisis e interpretación | 15 |
| Orden y documentación | 5 |
| **Total** | **100** |

---
*Universidad de Especialidades UNE · Plantel Centro · Optativa III: Ciencia de Datos*
*Datasets: ventas_sucias.csv, clientes.json, tienda.db*


# Guia de respuestas

## Parte A

1. La limpieza ocupa gran parte del proyecto porque los datos reales contienen nulos, duplicados, errores, formatos distintos y valores imposibles. Sin limpiarlos, el modelo aprenderia ruido y produciria resultados poco confiables.
2. El dataset original tiene **630 filas y 10 columnas**.
3. Tienen nulos `fecha` (93), `pais` (50), `categoria` (54), `precio_unitario` (51), `metodo_pago` (84), `calificacion` (94) y `edad_cliente` (78). La que mas tiene es `calificacion`, con 94.
4. Hay **30 filas duplicadas**. Inflan conteos, promedios y totales, por lo que el analisis queda sesgado.
5. `precio_total` se carga como texto porque algunos valores incluyen el simbolo `$` o espacios, y pandas no puede tratarlos directamente como numeros.
6. Se eliminaron **30 filas**. `reset_index(drop=True)` crea un indice consecutivo nuevo y evita conservar el indice de las filas eliminadas.
7. `df.drop_duplicates(subset=["id_venta"])` elimina duplicados usando solo `id_venta`.
8. Convierte a numero; con `errors="coerce"`, los valores imposibles se convierten en `NaN` en vez de provocar un error.
9. `astype(str)` permite aplicar operaciones de texto; `str.replace` elimina `$`; `str.strip` quita espacios al inicio y al final.
10. Si una conversion no es posible, aparece un nuevo nulo (`NaN`). Se debe contar con `df["precio_total"].isnull().sum()`.
11. Antes habia **10** variantes de pais y despues quedan **4**: Colombia, España, México y Perú. Unificarlas evita separar ventas del mismo pais.
12. `strip` quita espacios y `lower` convierte a minusculas. Primero se quitan espacios para que el texto se normalice correctamente.
13. Las ventas de un mismo pais quedarian repartidas en varias categorias y los totales por pais serian incorrectos.
14. Con el diccionario ampliado quedan **4 categorias**: Electrónica, Hogar, Ropa y Juguetes.
15. Un diccionario manual puede omitir variantes. Se puede automatizar con normalizacion de acentos, minusculas, similitud de texto y un catalogo de valores validos.
16. La mediana resiste mejor los valores extremos; la media subiria demasiado si hay precios atipicos.
17. Pais y categoria son variables categoricas, por eso se usa la moda. Una media de paises no tiene sentido.
18. Conviene eliminar cuando faltan pocos datos, la fila es critica o no existe una imputacion razonable. Si faltan muchos, eliminar puede perder demasiada informacion.
19. Repetir la misma mediana o moda reduce la variabilidad y puede introducir sesgo.
20. `fecha` tiene **90 nulos despues de quitar duplicados**. Elegiria conservar las filas y marcar la fecha como faltante si no es esencial; si se necesita analizar por tiempo, eliminaria esas filas o buscaria la fecha en la fuente original.
21. En `edad_cliente` aparecen -5 y 200; en `calificacion`, 99. Primero se convierten en `NaN` porque son datos invalidos y despues se imputan con una estadistica calculada solo con valores validos.
22. El 99 elevaria artificialmente el promedio y ocultaria la calificacion real de los clientes.
23. Una regla de negocio define valores aceptables. Para edad: entre 0 y 120 años. Para calificacion: entre 1 y 5.
24. Aplicando IQR despues de convertir los precios invalidos a `NaN` e imputar los faltantes, el limite superior es aproximadamente **1655.34** y se detectan **21 outliers**.
25. IQR es `Q3 - Q1`, el rango del 50% central. El umbral 1.5 × IQR identifica valores alejados sin depender demasiado de la media.
26. La caja conserva la zona central, pero los extremos quedan limitados. `clip(upper=...)` reemplaza los valores superiores al limite por ese limite.
27. Otra estrategia es eliminar outliers. Es preferible eliminarlos si son errores confirmados; el capping es mejor si pueden ser reales y no se quiere perder filas.
28. Una compra empresarial muy grande puede ser un outlier real y valioso; no deberia eliminarse sin investigar.
29. `pd.cut` divide una variable continua en intervalos y crea categorias ordenadas. `precio_unitario` conserva valores numericos; `rango_precio` agrupa esos valores.
30. Permite comparar `precio_total_calculado` con `precio_total` y detectar errores de cantidad, precio o captura.
31. Inicialmente habia 630 filas y 30 duplicados; despues quedan 600. Se corrigieron tipos, nulos, textos, edades, calificaciones y outliers. Quedan nulos de `fecha` porque no se imputaron.
32. Se grafica con `df["categoria"].value_counts().plot(kind="bar")`. La categoria con mas ventas se determina con `df["categoria"].value_counts().idxmax()` despues de normalizar.

## Parte B

33. Hay **150 clientes y 6 columnas**. Un CSV suele ser tabular; JSON puede contener objetos y estructuras anidadas.
34. Hay nulos en `nombre` (20), `email` (19) y `ciudad` (28).
35. Hay espacios, mayusculas inconsistentes y nombres vacios; tambien hay valores faltantes.
36. `.str.capitalize()` pone la primera letra en mayuscula y el resto en minusculas. Despues de limpiar aparecen Ana, Carlos, José, Luis, María y Sofía, ademas de 20 valores vacios que deben convertirse en `NaN`.
37. Hay **56 emails invalidos** porque no contienen `@`, incluyendo vacios y `None`.
38. Quedan **3 ciudades**: Guadalajara, Monterrey y Ciudad de México. Se unificaron `CDMX`/`cdmx` y `GDL`/`Guadalajara`.
39. Promedio de gasto: Ciudad de México **917.43**, Guadalajara **963.60** y Monterrey **1004.47**. Monterrey gasta mas en promedio.
40. Hay **80 productos**. Las columnas son `producto_id`, `nombre_producto`, `categoria`, `stock` y `precio`.
41. SQL permite consultar, filtrar e indexar grandes volumenes sin cargar toda la tabla en memoria y ofrece mejor control de integridad y concurrencia.
42. Hay **6 productos** con stock menor a 50. Filtrar en SQL reduce los datos transferidos y procesados por pandas.
43. La consulta es:

```python
resultado = pd.read_sql(
    "SELECT categoria, AVG(precio) AS precio_prom FROM productos GROUP BY categoria",
    conn
)
```

Los promedios son Electrónica 386.01, Hogar 299.46, Juguetes 431.23 y Ropa 331.89. La categoria mas cara es **Juguetes**.
44. `groupby().agg()` agrupa filas y calcula medidas. `num_ventas` cuenta `id_venta`; `precio_promedio` calcula la media de `precio_unitario`.
45. `merge` une tablas usando una clave comun. Permitiria responder, por ejemplo, cuanto compra cada cliente o que clientes generan mas ingresos.
46. La venta promedio se calcula con `df["precio_total"].mean()`, la calificacion con `df["calificacion"].mean()` y el metodo mas usado con `df["metodo_pago"].mode()[0]`. Deben reportarse los valores que muestre la celda despues de ejecutar toda la limpieza.
47. No. Los duplicados inflarian los totales, los nulos causarian perdida de filas, los textos impedirian promediar, el 99 distorsionaria la calificacion y los outliers elevarian el promedio de precios.
48. La validacion de rangos y la correccion de tipos son especialmente importantes: impiden que valores imposibles o textos alteren todos los calculos posteriores.
49. Limpiar datos es parte esencial de la ciencia de datos: sin esa etapa, las estadisticas, graficas y modelos describen errores de captura en vez del fenomeno real.
50. Un caso real son los logs de un servidor. Se esperarian fechas en formatos distintos, registros repetidos, campos vacios, errores de escritura, unidades inconsistentes y valores extremos de latencia o consumo.

### Retos de codigo

```python
# Reto 1
 df["categoria"] = df["categoria"].str.strip().str.lower()
mapa_cat = {
    "electronica": "Electrónica", "electrónica": "Electrónica",
    "ropa": "Ropa", "hogar": "Hogar", "juguetes": "Juguetes"
}
df["categoria"] = df["categoria"].replace(mapa_cat)

# Reto 2
print("Nulos en fecha:", df["fecha"].isnull().sum())
print("Formatos de ejemplo:", df["fecha"].dropna().unique()[:6])

# Reto 3
clientes["nombre"] = clientes["nombre"].fillna("").str.strip().str.capitalize()
clientes["nombre"] = clientes["nombre"].replace("", np.nan)

# Reto 4
clientes["ciudad"] = clientes["ciudad"].str.strip().str.upper()
mapa_ciudad = {"CDMX": "Ciudad de México", "GDL": "Guadalajara"}
clientes["ciudad"] = clientes["ciudad"].replace(mapa_ciudad)

# Reto 5
conn = sqlite3.connect("tienda.db")
bajo_stock = pd.read_sql("SELECT * FROM productos WHERE stock < 50", conn)
conn.close()
print("Productos con bajo stock:", len(bajo_stock))
```


### Correccion de la respuesta 24

Al convertir `precio_unitario` con `pd.to_numeric(..., errors="coerce")`, imputar los valores invalidos con la mediana y ejecutar el codigo del notebook, el resultado es: **Q1 = 299.88**, **Q3 = 789.66**, **limite superior = 1524.33** y **33 outliers**. Usa estos valores en la respuesta 24.

### Codigo corregido del Reto 1

```python
df["categoria"] = df["categoria"].str.strip().str.lower()
mapa_cat = {
    "electronica": "Electrónica", "electrónica": "Electrónica",
    "ropa": "Ropa", "hogar": "Hogar", "juguetes": "Juguetes"
}
df["categoria"] = df["categoria"].replace(mapa_cat)
```